# Financial Data Analysis & Visualization

This notebook processes accounting data (imported from Excel/CSV), computes financial metrics, and visualizes trends.

**Key Phases:**
1.  **Data Loading**: Scans `../data/etica` for `.xls` files, converts them, and merges them.
2.  **Processing**: Computes the cumulative balance (`Saldo`), performs OLS linear regression, and resamples data into candlesticks.
3.  **Analysis**: Filters specific investment transactions and calculates derivatives.
4.  **Visualization**: Generates static plots (Matplotlib) for balances/derivatives and interactive plots (Plotly) for candlesticks.

In [ ]:
from piggy_bank.etica import *

# Ensure plots display in the notebook
%matplotlib inline

init_logging(also_time=False)

print('Loading data from', DATA_DIR)

## 1. Data Loading & Preprocessing

This section handles the ingestion of raw financial data.

In [ ]:
_df_all = load_data()

print(f"Total rows in processed data: {_df_all.shape[0]}")
print(_df_all.head())

# Filter dataframes using the newly added 'Blacklisted' column
_df_net = _df_all[_df_all['Blacklisted'] == 0].copy().drop(columns=['Blacklisted'])
_df_blk = _df_all[_df_all['Blacklisted'] == 1].copy().drop(columns=['Blacklisted'])

print(f'Net entries: {_df_net.shape[0]}')
print(f'Blacklisted entries: {_df_blk.shape[0]}')

---
## 2. Financial Metrics & Computation

This section computes the core financial metrics:

* **`compute_importo`**: Creates the `Saldo` (cumulative balance) by summing Credits (`Avere`) and Debits (`Dare`).
* **`compute_ols`**: Performs an Ordinary Least Squares (OLS) linear regression on the balance over time to identify the trend (slope `m` and intercept `q`).
* **`compute_candlesticks`**: Resamples the raw transaction data into OHLC (Open, High, Low, Close) candlesticks for specific timeframes (Monthly/Weekly).

### Compute Balance and Regression

In [ ]:
_df_net = compute_importo(_df_net)
_df_net, _m_net, _q_net = compute_ols(_df_net)
print(_df_net)

In [ ]:
_df_all = compute_importo(_df_all)
_df_all, _m_mov, _q_mov = compute_ols(_df_all)
print(_df_all)

In [ ]:
_df_blk = compute_importo(_df_blk)
_df_blk, _m_blk, _q_blk = compute_ols(_df_blk)

## Main Plot: Saldo and OLS

In [ ]:
plot_saldo_ols(_df_net, _df_all)

In [ ]:
plot_saldo_ols_interactive(_df_net, _df_all)

---
# Candlesticks

### Monthly

Also the derivative of the monthly closing balance (Change in `Close`).

In [ ]:
# Compute. Use ME for Month End, W for Weekly
_cndl_type = 'ME'
_cndl_lbl = 'Monthly' if _cndl_type == 'ME' else 'Weekly' if _cndl_type == 'W' else _cndl_type
_df_net_candles = compute_candlesticks(_df_net, _cndl_type)
_df_all_candles = compute_candlesticks(_df_all, _cndl_type)
print(f'{_cndl_lbl} candlesticks DataFrame head:\n{_df_net_candles.head()}')

In [ ]:
plot_candlestick(_df_net_candles, _cndl_lbl)

In [ ]:
plot_candlestick_interactive(_df_net_candles, 'Net')

In [ ]:
plot_candlestick_interactive(_df_all_candles, 'All')

---
## 3. Investment Filtering & Analysis

This section focuses on isolating investment-related transactions.

* **`filter_investments`**: Filters rows containing keywords like 'ETICA', 'FONDI', or 'DEPOSIT'.
* **Synthetic Bias**: Appends synthetic rows (`BILANCIO INIZIALE`) to simulate an initial balance or adjustment (bias) for accurate cumulative tracking.

In [ ]:
_df_inv = filter_investments(_df_net)
print_investments_summary(_df_inv)

Now plot the investment balance over time, if any investment transactions were found.

This will show how the investment-related balance evolved, which can be useful for tracking performance or identifying trends in the investment activities.

In [ ]:
plot_investments(_df_inv)